# 256→128 Wavelet DVF：低容量PC用・段階検証

このノートブックは既存の学習ノートブック・チェックポイントを変更しません。出力は `low_memory_wavelet_check/` のみです。

- **A**：人工画像で一定移動を検証（データ不要、まずこれだけ実行）
- **B**：患者1症例だけで同じ一定移動を検証
- **C**：患者1症例だけで10 stepだけ学習し、DVF/画像損失の重みを確認

`Data/TrainData_NoBed.npz` は全症例を展開するため使用しません。B/Cでは1症例だけの `.npy` または `.npz` を指定してください。

In [ ]:
# ===== 実行設定：ここだけ変更します =====
# 今回はA（人工画像）とB（患者1症例）をまとめて確認します。
# C（短期学習）は、Bの結果を確認してから追加してください。
STAGES = ('A', 'B')

# B/Cで使う、1症例だけを含むファイル。'AUTO'ならData配下から自動選択します。
# TrainData_NoBed.npz は自動選択されません。
IMAGE_PATH = 'AUTO'
# 手動指定する場合の例：IMAGE_PATH = r'Data/Longitudinal22/pair1/registered_masked_A_....npz'
NPZ_KEY = None       # NPZに複数キーがある場合のみ指定
VOLUME_INDEX = 0     # 小さい4次元ファイルに複数症例がある場合のみ使用

SHORT_TRAIN_STEPS = 10      # Cは最初は10のまま
CHECKPOINT_PATH = None      # 任意：既存の事前学習済み重み。読み込むだけで上書きしません。
FORCE_CPU = False            # GPUのメモリが不足する場合のみ True
OUTPUT_DIR = 'low_memory_wavelet_check'


In [ ]:
# 必要な関数は、低容量PC向けの検証スクリプトから読み込みます。
# このノートブックと low_memory_wavelet_check.py を同じ Saito フォルダに置いてください。
from pathlib import Path
import torch
import low_memory_wavelet_check as check

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
device = torch.device('cpu' if FORCE_CPU or not torch.cuda.is_available() else 'cuda')
print(f'Device: {device}')
print(f'Output folder: {output_dir.resolve()}')
print(f'Stages: {STAGES}')


ModuleNotFoundError: No module named 'low_memory_wavelet_check'

In [ ]:
# ===== A/B/Cをまとめて実行するセル =====
# A：人工画像だけなので、全データ読込み・モデル学習はありません。
if 'A' in STAGES:
    print('\n===== Stage A: artificial constant-shift check =====')
    synthetic_moving = check.make_test_image((32, 64, 64), device)
    check.constant_shift_check(synthetic_moving, output_dir / 'stage_A_synthetic', device)

# B/Cは同じ1症例だけを一度読んで使います。
one_volume = None
if 'B' in STAGES or 'C' in STAGES:
    if IMAGE_PATH == 'AUTO':
        data_root = Path('Data')
        candidates = [
            path for path in data_root.rglob('*')
            if path.suffix.lower() in ('.npy', '.npz')
            and path.name != 'TrainData_NoBed.npz'
        ]
        # Moving/registered画像を優先して、最初の1症例だけを選ぶ。
        candidates.sort(key=lambda path: (
            0 if any(word in path.name.lower() for word in ('registered', 'moving', 'masked_a')) else 1,
            str(path),
        ))
        if not candidates:
            raise FileNotFoundError('Data配下に個別症例のNPY/NPZがありません。IMAGE_PATH を手動指定してください。')
        selected_image_path = candidates[0]
        print(f'AUTO selected one-volume file: {selected_image_path}')
    elif IMAGE_PATH is None:
        raise ValueError('Stage B/Cでは、IMAGE_PATH を指定するか AUTO にしてください。')
    else:
        selected_image_path = Path(IMAGE_PATH)
    one_volume = check.read_one_volume(selected_image_path, NPZ_KEY, VOLUME_INDEX, device)
    print(f'Loaded one volume: {tuple(one_volume.shape)}')

if 'B' in STAGES:
    print('\n===== Stage B: one-volume constant-shift check =====')
    check.constant_shift_check(one_volume, output_dir / 'stage_B_one_volume', device)

if 'C' in STAGES:
    print('\n===== Stage C: short loss-balance training =====')
    checkpoint = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else None
    check.short_training_check(one_volume, SHORT_TRAIN_STEPS, output_dir / 'stage_C_short_training', device, checkpoint)

print('\n完了しました。Stage Cを実行した場合は、最後に出る image/DVF の値とCSVを確認してください。')


## 判定

- **Stage A**：`x=2` の誤差が非常に小さければ、フル解像度2 voxel → Wavelet格子1 voxel の換算は確認できます。`x=1` はHaarの偶奇位相のため差が残るのが正常です。
- **Stage C**：`image/DVF = 1` が両損失の寄与が同程度です。10以上なら画像損失が支配的、0.1以下ならDVF損失が支配的です。

結果フォルダの `constant_shift_metrics.csv` と `short_training_loss_balance.csv` を確認し、数値を共有してください。